# Where does this object live? — a notebook DSL for categorical bookkeeping

Mathematical writing constantly moves objects between categories without
saying so. "Consider the group $G$ as a set." "Regard the ring $R$ as an
abelian group." "View this scheme as a topological space." Each of these
silently applies a functor — usually a forgetful or inclusion functor —
and the reader is trusted to track which one, and to know that composites
of such moves cohere. Informally this is harmless abuse of notation; in
formalized mathematics it becomes the notorious pain of *coercions*.

This notebook demonstrates a small **Lean-elaborated DSL** exploring a
middle way, built on three ideas:

1. **An object's home category is part of its declaration.**
   `let X := t ∈ C` is a *checked judgment*, not an annotation: it
   elaborates to a real Lean definition of type `Object C`, verified by
   Lean's kernel against mathlib's category theory.
2. **Transport conventions are declared once, as state.** `prefer F`
   registers a functor as *the* canonical way to move between two
   categories — a machine-checked analogue of a paper's "Notation and
   conventions" section.
3. **Transport questions are answered mechanically.** `#via X ∈ D` asks
   "how does $X$ become an object of $D$?" and the system finds a path
   through the registered functors by composition search.

Every code cell is ordinary **Lean 4** — the Jupyter kernel never
interprets any of it; the DSL commands are Lean *command elaborators*
loaded from the `NbDsl.Notebook` prelude. **All of mathlib is in scope**,
`sage.all`-style.

> Working note: Lean has no top-level term display, so a bare `G` on its
> own line is a syntax error, not an echo of the value. Inspect things
> with `#check G`, `#eval` (for computables), the DSL's `#home G`, or by
> hovering an identifier (Shift+Tab).

In [1]:
-- Cells are plain Lean, with all of mathlib loaded.
#eval 6 * 7
#check Nat.exists_infinite_primes

Starting Lean worker (/home/dzack/gitclones/lean-jupyter-kernel/dsls/nbdsl)…
2:0: 42
3:0: Nat.exists_infinite_primes (n : ℕ) : ∃ p, n ≤ p ∧ Nat.Prime p


## The ambient universe

The DSL's standard module ships two *bundled categories* (a category
packaged together with its objects and structure, in mathlib's sense):

- `Groups` — mathlib's `GrpCat`, groups and group homomorphisms;
- `Sets` — `Type 0` with ordinary functions.

For a bundled category `C`, the type `Object C` is its type of objects.
Opening the namespaces brings the universe into scope — and note that
`open`, like every scope effect here, persists across cells:

In [2]:
open NbDsl NbDsl.Std

## Declaring an object *with* its home

Let us declare the symmetric group $S_3$ — the permutations of three
letters, the smallest non-abelian group — as an object of `Groups`. In
mathlib, $S_3$ is `Equiv.Perm (Fin 3)`, and `GrpCat.of` bundles any type
carrying a `Group` instance into an object of the category:

In [3]:
let S3 := GrpCat.of (Equiv.Perm (Fin 3)) ∈ Groups

No output is good news: the cell elaborated to a genuine definition
`S3 : Object Groups`, accepted by Lean's kernel. The `∈ Groups` is a
**type ascription**, so membership is enforced rather than recorded —
try changing it to `∈ Sets` and the cell becomes a *type error*, because
a bundled group is not an element of `Object Sets`. (Cells are atomic:
an erroring cell commits nothing, so experiments like that are free.)

Because the home category is carried by the *type*, it can be read back
off the declaration itself — no side table involved:

In [4]:
#check S3
#home S3

1:0: S3 : Object Groups
2:0: S3 ∈ NbDsl.Std.Groups


## Conventions as document state

There are many functors `Groups ⥤ Sets` — the forgetful functor, the
one-point functor, $G \mapsto G \times G$, … . Mathematical practice
fixes *one* as the default reading of "a group, viewed as a set": the
forgetful functor $U$. The `prefer` command records that choice.

The registration lands in a **persistent Lean environment extension** —
the same mechanism that backs mathlib's `simp` sets. In a notebook that
has real consequences: the conventions are part of the document's
semantic state, so they roll back if their cell later fails, they are
reconstructed on kernel restart (replay or session cache), and they are
visible to tooling.

In [5]:
prefer groupsToSets

1:0: preferred: NbDsl.Std.groupsToSets : NbDsl.Std.Groups ⥤ NbDsl.Std.Sets


## Asking the transport question

Now "how is $S_3$ a set?" has a mechanical answer: `#via` searches for a
path from the object's home to the target category through the
registered functors (breadth-first, so a shortest composite) and reports
it — here, one step through $U$:

In [6]:
#via S3 ∈ Sets

1:0: S3 ∈ NbDsl.Std.Sets via NbDsl.Std.Groups → NbDsl.Std.Sets


S3 ∈ NbDsl.Std.Sets via NbDsl.Std.Groups → NbDsl.Std.Sets

The answer also arrives as a structured MIME bundle
(`application/vnd.nbdsl.path+json`) naming the object, source, target,
and each functor on the path — rendered as the chain above and
machine-readable for downstream tooling.

With a richer universe (rings, modules, spaces, …) this becomes genuine
path search through a diagram of categories, and the real mathematical
content surfaces: *different* composites between the same endpoints need
not agree, so what the registry encodes is a chosen **coherent system of
transport conventions** — the thing a careful paper fixes in its
preamble and readers are otherwise left to reconstruct.

## Predicates as methods of a category

Mathematical properties — abelian, finite, simple, connected — are
*predicates on the objects of a category*, and any property worth having
is isomorphism-invariant. Categorically: it is the object part of a
functor $\mathrm{core}(\mathcal{C}) \to \mathrm{Prop}$, where passing to
the core (keeping only isomorphisms) is exactly what makes functoriality
*be* iso-invariance. The `predicate` command declares such a method in
the notebook — the POC records the object part as a kernel-checked
definition `Object C → Prop` and registers it as a method of `C`; the
iso-invariance (morphism-part) proof obligation is the natural next
refinement.

The binder coerces to the object's carrier, so the body is ordinary
element-level mathematics:

In [7]:
predicate Abelian (G ∈ Groups) := ∀ a b : G, a * b = b * a

4:0: s3_nonabelian : ∃ a b, a * b ≠ b * a


In [ ]:
-- The category now knows its methods:
#methods Groups

Now check the property on *specific declared objects*. `decide` performs
the exhaustive finite check and produces a proof term the kernel accepts.
For contrast, declare $S_2$ as well — the symmetric groups straddle the
commutativity boundary exactly between $n = 2$ and $n = 3$:

In [ ]:
let S2 := GrpCat.of (Equiv.Perm (Fin 2)) ∈ Groups

In [ ]:
theorem s3_nonabelian : ¬ Abelian S3 := by decide

example : Abelian S2 := by decide

#check s3_nonabelian

## What the system refuses to do

Transport is **directional and explicit**. Declare a plain set:

In [8]:
let B := Bool ∈ Sets

Asking how $B$ is a *group* has a perfectly good mathematical answer —
the free group functor, left adjoint to $U$ — but no such convention has
been registered, and the system will not guess on your behalf. The final
cell therefore **fails on purpose**, and (atomicity again) commits
nothing. Registering a preferred free-group functor is the natural next
extension of the universe, and would make this very query resolve.

In [9]:
-- Deliberately fails: no registered path Sets → Groups.
#via B ∈ Groups

LeanError: no preferred-functor path from NbDsl.Std.Sets to NbDsl.Std.Groups